# Exploratory Data Analysis (EDA)
## Predicting Customer Lifetime Value to Optimize Value-Based Bidding and Paid Advertising Budget Allocation

**Student:** Christian Alan Vibar\
**Course:** MAB2134 Analytics Algorithms 1 (Predictive Analytics 1)  
**Term:** 3rd / SY 2025–2026  
**Instructor:** Francis Adrian Viernes, CFA, MBA, MSF, CCREP, PMDSA

---

### Purpose of this notebook
This notebook covers the Data Understanding phase of CRISP-DM for the project. It documents the target distribution, feature distributions, data quality audit, bivariate relationships, and segment analysis. Findings from this EDA directly inform the preprocessing plan and modeling hypotheses for the project.

---

### Data Overview

**Source:** Internal company data warehouse (production gold layer). PII scrubbed prior to use.  
**Cohort:** Customers acquired between the years 2020 and 2026. All records have been ensured to have a complete 12-month revenue observation window as of the pull date.  
**Pull date:** 05/05/2026\
**Data Dictionary:** [Access in Github](https://github.com/christianvibar/mab2134_algo1/blob/5bcf33e33eee997443789d7bad652647520dc872/data_dictionary.md)\
**File used:** `mab2134_data_raw.csv`

The data was extracted using the following query:

```sql
SELECT 
    CONCAT(unified_client_id, '_', lifecycle) AS client_id,
    hubspot_client_id,
    lifecycle,
    channel,
    broader_source,
    broad_source,
    service_offering,
    campaign_group,
    campaign,
    ad_group,
    ad,
    ad_network,
    creative_group,
    creative_variation,
    keyword_text,
    keyword_match_type,
    placement,
    facebook_ad_set_audience,
    client_country,
    client_location_state,
    client_local_timezone,
    cb_city,
    email_type,
    email_industry_type,
    cb_jobtitle_type,
    cb_employees_range,
    cb_annual_revenue_ranges,
    cb_industry,
    cb_sector,
    cb_industry_group,
    cb_sub_industry,
    attributed_at,
    DATE(attributed_at) AS attributed_date,
    signup_at,
    signup_date,
    customer_date,
    first_billing_week,
    last_billing_week,
    revenue_first_1_month,
    revenue_first_2_months,
    revenue_first_3_months,
    revenue_first_6_months,
    total_revenue,
    starting_mrr,
    latest_mrr,
    profit,
    active_weeks,
    revenue_first_12_months
FROM `fivetran-magic-warehouse-fmdl.dbt_production.dim_clients_multi_lcs`
WHERE fivetran_deleted = FALSE
    AND invalid_contact_flag = FALSE
    AND is_merged = FALSE
    AND revenue_first_12_months IS NOT NULL
    AND active_weeks >= 51
```

> **Note on reproducibility:** The source data is proprietary company data and was retrieved via the organization's data warehouse. The SQL query above is purely for documentation. All analysis in this notebook is reproducible from the CSV file given the query above.

---
## 0. Preliminaries

In [ ]:
# BASIC PACKAGES
%matplotlib auto
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# REPRODUCIBILITY
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# DISPLAY OPTIONS
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 100)

# AESTHETICS
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = 'Blues_d'

print('Packages loaded successfully.')

## 0.1 Helper Functions

In [ ]:
def summarize_categoricals(df, cols):
    """
    Prints a cardinality and missingness summary for a list of categorical columns,
    sorted by unique values descending.
    """
    cols = [c for c in cols if c in df.columns]
    summary = pd.DataFrame({
        'Unique Values': [df[c].nunique(dropna=True) for c in cols],
        'Missing %': [(df[c].isnull().mean() * 100).round(1) for c in cols]
    }, index=cols).sort_values('Unique Values', ascending=False)

    print('=== Categorical Feature Summary ===')
    print(summary.to_string())

def plot_categorical(df, col):
    """
    Horizontal bar chart for a single categorical feature.
    Shows value counts with percentage labels.
    """
    counts = df[col].value_counts(dropna=False)
    pct = counts / len(df) * 100

    fig, ax = plt.subplots(figsize=(12, max(3, len(counts) * 0.4)))
    sns.barplot(x=counts.values, y=counts.index.astype(str), ax=ax, color='steelblue')
    ax.set_title(f'{col}  |  {df[col].nunique()} unique values', fontweight='bold')
    ax.set_xlabel('Count')
    ax.set_ylabel('')
    for i, (count, p) in enumerate(zip(counts.values, pct)):
        ax.text(count + 1, i, f'{p:.1f}%', va='center', fontsize=9)
    plt.tight_layout()


def plot_numeric(df, col):
    """
    Histogram + box plot side by side for a single numeric feature.
    Prints skewness below the plots.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    sns.histplot(df[col].dropna(), bins=40, ax=axes[0], color='steelblue')
    axes[0].set_title(f'{col} — Distribution', fontweight='bold')
    axes[0].set_xlabel(col)
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    sns.boxplot(x=df[col].dropna(), ax=axes[1], color='steelblue')
    axes[1].set_title(f'{col} — Box Plot', fontweight='bold')
    axes[1].set_xlabel(col)
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    plt.tight_layout()
    print(f'Skewness: {df[col].skew():.3f}')


def plot_clv_by_category(df, col, target):
    """
    Box plot + median bar chart side by side for a categorical feature vs. target.
    Groups are sorted by median target value descending.
    """
    order = df.groupby(col)[target].median().sort_values(ascending=False).index

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    sns.boxplot(data=df, x=col, y=target, order=order, ax=axes[0], palette='Blues')
    axes[0].set_title(f'12-Month Revenue by {col}', fontweight='bold')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Revenue (USD)')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    axes[0].tick_params(axis='x', rotation=30)

    medians = df.groupby(col)[target].median().sort_values(ascending=False)
    sns.barplot(x=medians.index, y=medians.values, ax=axes[1], palette='Blues_d')
    axes[1].set_title(f'Median 12-Month Revenue by {col}', fontweight='bold')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Median Revenue (USD)')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    axes[1].tick_params(axis='x', rotation=30)

    plt.tight_layout()
    print(df.groupby(col)[target].agg(['count', 'median', 'mean', 'std']).round(0))


def plot_clv_scatter(df, col, target):
    """
    Scatter plot of a numeric feature vs. target with Pearson correlation annotation.
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.scatter(df[col], df[target], alpha=0.3, color='steelblue', s=15)
    ax.set_title(f'{col} vs. 12-Month Revenue', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Revenue (USD)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    corr = df[[col, target]].dropna().corr().iloc[0, 1]
    ax.annotate(f'r = {corr:.3f}', xy=(0.05, 0.93), xycoords='axes fraction', fontsize=11,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

    plt.tight_layout()

def plot_segment_heatmap(df, row_col, col_col, target):
    """
    Two heatmaps side by side: customer count and median target value
    for a row_col × col_col segment cross-tabulation.
    """
    cross_count = pd.crosstab(df[row_col], df[col_col])
    cross_rev = df.groupby([row_col, col_col])[target].median().unstack()

    fig, axes = plt.subplots(1, 2, figsize=(18, max(4, len(cross_count) * 0.5)))

    sns.heatmap(cross_count, annot=True, fmt='d', cmap='Blues',
                ax=axes[0], linewidths=0.5)
    axes[0].set_title(f'Customer Count: {row_col} × {col_col}', fontweight='bold')
    axes[0].set_xlabel(col_col)
    axes[0].set_ylabel(row_col)

    sns.heatmap(cross_rev, annot=True, fmt='.0f', cmap='Blues',
                ax=axes[1], linewidths=0.5)
    axes[1].set_title(f'Median 12-Month Revenue: {row_col} × {col_col}', fontweight='bold')
    axes[1].set_xlabel(col_col)
    axes[1].set_ylabel(row_col)

    plt.tight_layout()


def compute_geometry(candidate_features, df):
    categorical_features = [f for f in candidate_features 
                             if df[f].dtype == 'object' or df[f].dtype.name == 'category']

    cardinality = df[categorical_features].nunique(dropna=False)

    summary = pd.DataFrame({
        'Type': ['Categorical'] * len(categorical_features) + 
                ['Numeric'] * (len(candidate_features) - len(categorical_features)),
        'Unique Values (approx OHE cols)': list(cardinality.values) + 
                [1] * (len(candidate_features) - len(categorical_features))
    }, index=categorical_features + 
              [f for f in candidate_features if f not in categorical_features])

    summary['Cumulative OHE Columns'] = summary['Unique Values (approx OHE cols)'].cumsum()

    total_ohe_cols = summary['Unique Values (approx OHE cols)'].sum()
    active_cols = len(candidate_features)
    sparsity = 1 - (active_cols / total_ohe_cols)

    print(summary.to_string())
    print(f"\nTotal candidate features (raw): {len(candidate_features)}")
    print(f"Estimated total columns (post-OHE): {total_ohe_cols}")
    print(f"Active columns per row (pre-OHE): {active_cols}")
    print(f"Estimated sparsity: {sparsity:.1%}")

    return summary, sparsity


def plot_design_matrix(candidate_features, df, sample_n=50, max_cols=80):
    categorical_features = [f for f in candidate_features 
                             if df[f].dtype == 'object' or df[f].dtype.name == 'category']

    df_candidates = df[candidate_features].copy()
    for col in categorical_features:
        df_candidates[col] = df_candidates[col].fillna('missing')

    df_encoded = pd.get_dummies(df_candidates, columns=categorical_features, dtype=int)

    numeric_features = [f for f in candidate_features if f not in categorical_features]
    ohe_cols = [c for c in df_encoded.columns
                if not any(c.startswith(n) for n in numeric_features)]

    total_cols = df_encoded.shape[1]
    sparsity = 1 - (len(candidate_features) / total_cols)

    df_sample = df_encoded[ohe_cols].iloc[:sample_n, :max_cols]

    fig, ax = plt.subplots(figsize=(20, 8))
    sns.heatmap(df_sample, cmap='Blues', cbar=True, linewidths=0.3, ax=ax)
    ax.set_title(f'Design Matrix — First {sample_n} Rows (post-OHE)\n'
                 f'{total_cols} columns | {sparsity:.1%} estimated sparsity',
                 fontweight='bold')
    ax.set_xlabel('Encoded Features')
    ax.set_ylabel('Row Index')
    ax.set_xticks([])
    plt.tight_layout()

def categorical_clv_summary(df, cat_cols, target):
    """
    For each categorical feature, computes group count, median CLV per group,
    a dollar gap (max - min group median), and a median spread ratio 
    (max group median / min group median).
    Higher spread ratio = more separation between groups = stronger candidate signal.
    """
    rows = []
    for col in cat_cols:
        valid = df[[col, target]].dropna()
        group_medians = valid.groupby(col)[target].median()
        min_med = group_medians.min()
        max_med = group_medians.max()
        rows.append({
            'Feature': col,
            'Groups': valid[col].nunique(),
            'Min Group Median ($)': round(min_med, 0),
            'Max Group Median ($)': round(max_med, 0),
            'Dollar Gap ($)': round(max_med - min_med, 0),
            'Spread Ratio': round(max_med / min_med, 2)
                           if min_med > 0 else None
        })
    return (pd.DataFrame(rows)
              .sort_values('Spread Ratio', ascending=False)
              .reset_index(drop=True))

def pairwise_interaction(df, features, target, min_n=10):
    """
    For all pairwise combinations of features, computes:
    - Number of populated cells (n >= min_n)
    - Median CLV per cell
    - Variance of cell medians (higher = more interesting interaction)
    - Dollar gap across cells (max cell median - min cell median)
    
    Returns a summary DataFrame ranked by cell median variance descending.
    """
    rows = []
    for f1, f2 in combinations(features, 2):
        valid = df[[f1, f2, target]].dropna()
        grouped = valid.groupby([f1, f2])[target].agg(['median', 'count']).reset_index()
        reliable = grouped[grouped['count'] >= min_n]
        
        if len(reliable) < 2:
            continue
            
        cell_medians = reliable['median']
        rows.append({
            'Feature 1': f1,
            'Feature 2': f2,
            'Total Cells': len(grouped),
            'Reliable Cells (n≥10)': len(reliable),
            'Min Cell Median ($)': round(cell_medians.min(), 0),
            'Max Cell Median ($)': round(cell_medians.max(), 0),
            'Dollar Gap ($)': round(cell_medians.max() - cell_medians.min(), 0),
            'Cell Median Variance': round(cell_medians.var(), 0)
        })
    
    return (pd.DataFrame(rows)
              .sort_values('Cell Median Variance', ascending=False)
              .reset_index(drop=True))

print('Helper functions loaded.')

---
## 1. Data Loading & Initial Inspection

In [ ]:
# LOAD DATA
FILE_PATH = '../data/mab2134_data_raw.csv'

df_orig = pd.read_csv(FILE_PATH)

# Always work on a copy; keep original untouched for reference
df = df_orig.copy()

print(f'File loaded: {FILE_PATH}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# DTYPES AND NULL SUMMARY
df.info()

In [ ]:
# FIRST ROWS
df.head()

In [ ]:
# DESCRIPTIVE STATISTICS
df.describe(include='all').T

### 1.1 Column Name Cleanup

In [ ]:
# Standardize column names: strip whitespace, lowercase
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print('Cleaned column names:')
print(df.columns.tolist())

### 1.2 Column Value Cleanup

In [ ]:
# Standardize column values: strip whitespace, lowercase
str_cols = df.select_dtypes(include='object').columns.tolist()

for col in str_cols:
    df[col] = df[col].str.strip().str.lower().str.replace(' ', '_', regex=False)

# Convert lifecycle to string - rationale to be explained in 4. Feature Analysis
df['lifecycle'] = df['lifecycle'].astype(str)

# Print head to check again
df.head()

**Date Parsing Corruption in `cb_employees_range`:** The values `1-10` and `11-50` were corrupted to `01-oct` and `nov-50` respectively, likely due to Excel auto-formatting numeric ranges as dates during data export. The correct values are restored below. This fix will also be applied in the preprocessing pipeline.

In [ ]:
# Fix Excel date-parsing corruption in cb_employees_range
df['cb_employees_range'] = df['cb_employees_range'].replace({
    '01-oct': '1-10',
    'nov-50': '11-50'
})

print(df['cb_employees_range'].value_counts(dropna=False))

### 1.3 Summary Snapshot (Full Dataset)

| Dimension | Value |
|---|---|
| Rows | *1273* |
| Columns | *48* |
| Target | `revenue_first_12_months` |

> **Observations:** Initial inspection shows that all rows have the target variable `revenue_first_12_months`. The minimum active_weeks is 51 (active weeks start at 0), ensuring that all clients in this analysis have been customers for at least 1 year. There are missing values in several independent variables that will be investigated in the Data Quality Audit section, but most notably the variables `service_offering`, `placement`, `keyword_text`, and `keyword_match_type` appears to be severely lacking.
> 
> Some things to flag on clean-up are text rendering issues from the csv upload. For example, the `cb_employees_range` value is showing up as Nov-50, when it was originally 11-50. The date and timestamp fields are also being rendered as objects right now, and would need to be transformed into the appropriate types.

### 1.4 Scope Filter: Paid Inbound Customers Only

The business objective is to predict CLV to guide **paid advertising budget allocation and value-based bidding**. The model will only ever be applied to customers acquired through paid channels. Organic, outbound, and referral customers do not enter the paid bidding pipeline and are not the population we need to predict for.

To align the training population with the inference population, we filter `df` to `broad_source == 'paid_inbound'` before all further analysis. All distributions, quality audits, and feature analyses in this notebook reflect this scoped population.

The full dataset (n=1,273) is retained in `df_orig` and `df_full` for reference.

In [ ]:
# Retain full dataset for reference
df_full = df.copy()

# Scope to paid inbound only — model training and inference population
df = df[df['broad_source'] == 'paid_inbound'].copy().reset_index(drop=True)

print(f"Full dataset:        {len(df_full):,} rows")
print(f"Paid inbound only:   {len(df):,} rows")
print(f"Rows removed:        {len(df_full) - len(df):,}")
print(f"\nbroad_source distribution after filter:")
print(df['broad_source'].value_counts())

### 1.5 Scoped Population Snapshot

| Dimension | Value |
|---|---|
| Rows (modeling population) | *535* |
| Rows removed (non-paid inbound) | *738* |
| Columns | *48* |
| Target | `revenue_first_12_months` |

> All subsequent analysis reflects the paid inbound population only (n=535).

---
## 2. Target Variable Analysis

Understanding the distribution of `revenue_first_12_months` before anything else. This is what the model will predict.

In [ ]:
TARGET = 'revenue_first_12_months'

# Basic stats
print('=== Target Variable Stats ===')
print(df[TARGET].describe())
print(f'\nZero values: {(df[TARGET] == 0).sum():,}')
print(f'Negative values: {(df[TARGET] < 0).sum():,}')
print(f'Null values: {df[TARGET].isnull().sum():,}')

In [ ]:
# DISTRIBUTION PLOT — raw and log-transformed side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw
sns.histplot(df[TARGET].dropna(), bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Target Distribution (Raw)', fontweight='bold')
axes[0].set_xlabel('12-Month Gross Revenue (USD)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Log-transformed (only positive values)
log_target = np.log1p(df[TARGET].dropna())
sns.histplot(log_target, bins=50, ax=axes[1], color='steelblue')
axes[1].set_title('Target Distribution (log1p)', fontweight='bold')
axes[1].set_xlabel('log1p(12-Month Gross Revenue)')
axes[1].set_ylabel('Count')

plt.tight_layout()

# Skewness check
print(f'Skewness (raw): {df[TARGET].skew():.3f}')
print(f'Skewness (log1p): {log_target.skew():.3f}')

In [ ]:
# PERCENTILE TABLE
percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
pct_table = df[TARGET].quantile(percentiles).reset_index()
pct_table.columns = ['Percentile', 'Revenue (USD)']
pct_table['Percentile'] = pct_table['Percentile'].apply(lambda x: f'{x:.0%}')
print(pct_table.to_string(index=False))

### 2.1 Target Variable Observations

There are no null rows nor negative values in the target variable, ensuring that the data is valid for analysis.

When it comes to the raw distribution of the target variable, it is clear that the data is right-skewed. This is supported by the distribution plot, the mean (`$23,574`) exceeding the median (`$17,620`) by `$5,954`, and a skewness value of 3.923. This indicates it is very far from 0 (perfect symmetry). Further, the 99th percentile (`$115,554`) is 6.5x the median, confirming that a small number of high-value accounts are pulling the distribution rightward. This becomes important especially when training regression models, which assume that residuals are roughly symmetric. In its current state, the model may be dominated by these outlier accounts, which could lead to poor performance.

However, transforming the data into a log scale eliminates the skew, bringing it to -0.525 (almost negligible). Practically speaking, we will most likely have to predict using `log1p(revenue)`. This means that all evaluation of the metrics will be measured in log units and not dollars. This is important to keep note of in the modelling phase of the project. The outputs then have to be converted back to dollars before presenting to stakeholders and/or deployment.

Finally, a key note in this target variable is that this model only shows all clients who have survived at least 51 weeks or 1 year. It doesn't capture all clients who churned in weeks 1 - 50. This is okay for our use case, as we are only interested to see which customer segments tend to be more valuable so that they can be used in value based bidding in ad platforms.

---
## 3. Data Quality Audit

Three checks: missingness, duplicates, and leakage audit.

### 3.1 Missingness

In [ ]:
# Missing value summary, sorted descending
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print('No missing values found.')
else:
    print(missing_df)

In [ ]:
# Visualize missingness
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_df) * 0.5)))
    sns.barplot(x='Missing %', y=missing_df.index, data=missing_df, color='steelblue', ax=ax)
    ax.set_title('Missingness by Feature (%)', fontweight='bold')
    ax.set_xlabel('% Missing')
    ax.set_ylabel('Feature')
    ax.axvline(40, color='red', linestyle='--', label='40% threshold')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

### 3.2 Duplicates

In [ ]:
# Full-row duplicates
dup_count = df.duplicated().sum()
print(f'Full-row duplicates: {dup_count:,}')

### 3.3 Leakage Audit

Every feature must be observable **at acquisition time** — before any retention, upsell, or account management action is taken. Features that reveal post-acquisition behavior are leaky.

| Feature | Observable at Acquisition? | Disposition |
|---|---|---|
| `client_id` | . | ID. Drop before modeling |
| `hubspot_client_id` | . | ID. Drop before modeling |
| `lifecycle` | Yes | Candidate feature. Numeric codes (1–5) representing customer type; cast to string before modeling to prevent ordinal misinterpretation |
| `channel` | Yes | Candidate feature. Attribution captured at acquisition |
| `broader_source` | Yes | Drop. This will only retain one value `paid_inbound`, since that is the only source we're analyzing for the project. |
| `broad_source` | Yes | Drop. This value is similar to `broader_source` when it concerns paid inbound traffic and thus not needed for analysis. |
| `service_offering` | Yes | Candidate feature. Service offering the client saw in advertising campaign. |
| `campaign_group` | Yes | Candidate feature. Attribution captured at acquisition |
| `campaign` | Yes | Candidate feature. Attribution captured at acquisition |
| `ad_group` | Yes | Candidate feature. Attribution captured at acquisition |
| `ad` | Yes | Candidate feature. High cardinality; likely too granular to use |
| `ad_network` | Yes | Candidate feature. Attribution captured at acquisition |
| `creative_group` | Yes | Candidate feature. High cardinality; likely too granular to use |
| `creative_variation` | Yes | Candidate feature. High cardinality; likely too granular to use |
| `keyword_text` | Yes | Drop. 72% missing; too granular |
| `keyword_match_type` | Yes | Drop. 74% missing; redundant with `channel` |
| `placement` | Yes | Drop. 99% missing |
| `facebook_ad_set_audience` | Yes | Candidate feature. Only populated for Facebook traffic; treat nulls as separate category or drop |
| `client_country` | Yes | Candidate feature. Known at signup |
| `client_location_state` | Yes | Candidate feature. Known at signup |
| `client_local_timezone` | Yes | Low signal. Likely redundant with geo fields; EDA to decide |
| `cb_city` | Yes | Clearbit enrichment at lead stage. High cardinality; likely drop |
| `email_type` | Yes | Candidate feature. Enrichment at lead stage |
| `email_industry_type` | Yes | Drop. 90% missing |
| `cb_jobtitle_type` | Yes | Candidate feature. Clearbit enrichment at lead stage |
| `cb_employees_range` | Yes | Candidate feature. Clearbit enrichment at lead stage; 26% missing; treat nulls as separate category |
| `cb_annual_revenue_ranges` | Yes | Candidate feature. Clearbit enrichment at lead stage; 27% missing; treat nulls as separate category |
| `cb_industry` | Yes | Candidate feature. Clearbit; 18% missing; treat nulls as separate category |
| `cb_sector` | Yes | Candidate feature. Clearbit; likely redundant with `cb_industry` |
| `cb_industry_group` | Yes | Candidate feature. Clearbit; likely redundant with `cb_industry` |
| `cb_sub_industry` | Yes | Candidate feature. Clearbit; high cardinality |
| `starting_mrr` | Yes | Candidate feature. Reflects plan/deal configuration at first billing; 155 unique values suggest mix of clean plan tiers and deal-level variance; keep as numeric; missingness (7.4%) to be addressed in preprocessing |
| `attributed_at` / `attributed_date` | . | Timestamp. Do not use raw; may extract acquisition year/month as cohort control variable |
| `signup_at` / `signup_date` | . | Timestamp. Same as above |
| `customer_date` | . | Timestamp. Drop; post-acquisition event |
| `first_billing_week` | . | Timestamp. Drop; `starting_mrr` already captures initial billing information |
| `last_billing_week` | No | Leaky. Encodes how long the customer stayed; directly derived from the same survival window as the target |
| `revenue_first_1_month` | No | Leaky. Partial cumulative revenue window; sub-component of the target |
| `revenue_first_2_months` | No | Leaky. Same as above |
| `revenue_first_3_months` | No | Leaky. Same as above |
| `revenue_first_6_months` | No | Leaky. Same as above |
| `total_revenue` | No | Leaky. Post-acquisition; encompasses the full customer lifetime beyond 12 months |
| `latest_mrr` | No | Leaky. Reflects post-acquisition account evolution |
| `profit` | No | Leaky. Derived from revenue, which includes the target period |
| `active_weeks` | No | Leaky. Used as a filter criterion (>= 51), not a predictor; directly encodes survival duration |
| `revenue_first_12_months` | Target | Target variable. Excluded from features |

##### 3.3.1 Starting MRR Inspection <a id="starting_mrr_inspect"></a>

`starting_mrr` requires closer inspection before classifying it in the leakage audit. As the company is subscription-based, first billing week MRR should reflect plan configuration at acquisition rather than post-acquisition behavior. The check below confirms whether the distribution supports this.

In [ ]:
# Value counts and basic stats
print('=== starting_mrr value counts (top 20) ===')
print(df['starting_mrr'].value_counts(dropna=False).head(20))
print(f'\nUnique values: {df["starting_mrr"].nunique()}')
print(f'Missing: {df["starting_mrr"].isnull().sum()} ({df["starting_mrr"].isnull().mean():.1%})')

plot_numeric(df, 'starting_mrr')

**Observation:** `starting_mrr` has 87 unique values across 535 rows (36 missing, 6.7%). The most frequent values (433, 866, 1,299, 1,732, 2,598) are clean multiples of a base rate, consistent with standard plan tiers. However, there are large outliers extending to `~$80,000` that warrant a closer look.

In [ ]:
# How many records exceed a reasonable plan ceiling?
thresholds = [5000, 10000, 20000]

for t in thresholds:
    n = (df['starting_mrr'] > t).sum()
    pct = n / len(df) * 100
    print(f'starting_mrr > ${t:,}: {n} rows ({pct:.1f}%)')

# See the actual outlier records
print('\n=== Records with starting_mrr > $10,000 ===')
print(df[df['starting_mrr'] > 10000][['lifecycle', 'channel', 'service_offering', 'starting_mrr', 'revenue_first_12_months']].sort_values('starting_mrr', ascending=False))

**Observation:** A small number of records (2.8%) show `starting_mrr` above `$10,000`, extending to `$93,571`. Inspection of these records shows that `revenue_first_12_months` tracks proportionally in most cases, suggesting these are legitimate large accounts rather than data errors. The main exception is row 194 (`starting_mrr` `$93,571`, `revenue_first_12_months` `$21,968`), where revenue is anomalously low relative to the plan size — this warrants review. Row 128 shows the inverse pattern (`starting_mrr` `$42,261`, `revenue_first_12_months` `$205,212`), which may reflect a legitimate high-retention account but will also be flagged for review. Outlier treatment strategy for `starting_mrr` will be decided in the preprocessing phase.

Additionally, missingness (6.7%) will be addressed in the preprocessing phase.

### 3.4 Redundant Constructs

Several features in the candidate set measure the same underlying construct at different levels of granularity. Retaining all of them would inflate the feature space without adding independent signal. For each construct group, one representative feature is selected based on granularity fit, cardinality, and coverage.

| Construct | Candidate Fields | Retained | Rationale |
|---|---|---|---|
| Acquisition channel | `channel`, `broad_source`, `broader_source`, `ad_network` | `ad_network` | After scoping to paid inbound customers, `broad_source` becomes constant. `ad_network` directly identifies the paid platform (adwords, facebook, bing) at the right level of granularity. `channel` is the raw attribution field and is superseded by `ad_network`. |
| Ad attribution | `campaign`, `ad_group`, `ad`, `creative_group`, `creative_variation` | `campaign_group` | All fields measure ad creative or placement at increasingly granular levels. `campaign_group` is the highest-level, most generalizable representation. |
| Geography | `client_country`, `client_location_state`, `client_local_timezone`, `cb_city` | `client_country` | State and city are too granular; timezone is redundant with country-level geography. |
| Industry classification | `cb_industry`, `cb_industry_group`, `cb_sub_industry`, `cb_sector` | `cb_sector` | All fields measure company industry at sub-sector granularity. `cb_sector` is the cleanest, lowest-cardinality representation. |

### 3.5 Data Quality Summary

**High-missingness columns (drop before modeling)**: `placement` (99.8%), `email_industry_type` (89.7%), `keyword_match_type` (41.9%), and `keyword_text` (37.9%) are from platform-specific ad fields. Their signal is already captured by `channel` and `ad_network`, and their missingness makes them unusable. These will be dropped.

**Leaky columns (drop before modeling)**: `revenue_first_1_month`, `revenue_first_2_months`, `revenue_first_3_months`, `revenue_first_6_months`, `total_revenue`, `profit`, `latest_mrr`, `last_billing_week`, and `active_weeks` encode post-acquisition behavior. These must be excluded before any feature engineering or model training.

**ID and timestamp columns (drop before modeling)**: `client_id`, `hubspot_client_id`, `attributed_at`, `attributed_date`, `signup_at`, `signup_date`, `customer_date`, and `first_billing_week` serve no predictive role as raw values. Acquisition year or month may be extracted from `attributed_at` as a cohort control variable if needed.

**High-missingness candidate features (retain with strategy)**: `cb_annual_revenue_ranges` (31.2% missing) and `cb_employees_range` (30.5% missing) are firmographic signals; nulls will be treated as a separate category. `starting_mrr` (6.7% missing) will be addressed in the preprocessing phase.

**On `starting_mrr`**: The raw MRR value from the first billing week reflects plan and deal configuration at acquisition (the company is a subscription business). With 87 unique values, it captures a mix of clean plan tiers and deal-level variance (multi-assistant configurations, prorations). It will be retained as a numeric feature. Missingness and outliers will be addressed in the preprocessing phase.

**Duplicates**: No full-row duplicates were found.

**Target variable**: `revenue_first_12_months` has no nulls and no negative values, confirming data validity for regression.

After all drops, the 12 remaining modeling columns are:

| # | Column | Role |
|---|---|---|
| 1 | `lifecycle` | Feature |
| 2 | `service_offering` | Feature |
| 3 | `campaign_group` | Feature |
| 4 | `ad_network` | Feature |
| 5 | `client_country` | Feature |
| 6 | `email_type` | Feature |
| 7 | `cb_jobtitle_type` | Feature |
| 8 | `cb_employees_range` | Feature |
| 9 | `cb_annual_revenue_ranges` | Feature |
| 10 | `cb_sector` | Feature |
| 11 | `starting_mrr` | Feature |
| 12 | `revenue_first_12_months` | Target |

In [ ]:
# Apply all drop decisions from Section 3
# High-missingness columns
HIGH_MISSINGNESS_DROPS = [
    'placement', 'email_industry_type', 'keyword_match_type', 'keyword_text'
]

# Leaky columns
LEAKY_DROPS = [
    'revenue_first_1_month', 'revenue_first_2_months', 'revenue_first_3_months',
    'revenue_first_6_months', 'total_revenue', 'profit', 'latest_mrr',
    'last_billing_week', 'active_weeks'
]

# ID and timestamp columns
ID_DROPS = [
    'client_id', 'hubspot_client_id', 'attributed_at', 'attributed_date',
    'signup_at', 'signup_date', 'customer_date', 'first_billing_week'
]

# Redundant construct columns (Section 3.4)
REDUNDANT_DROPS = [
    'channel', 'broad_source', 'broader_source',
    'campaign', 'ad_group', 'ad', 'creative_group', 'creative_variation',
    'facebook_ad_set_audience',
    'client_location_state', 'client_local_timezone', 'cb_city',
    'cb_industry', 'cb_industry_group', 'cb_sub_industry'
]

ALL_DROPS = HIGH_MISSINGNESS_DROPS + LEAKY_DROPS + ID_DROPS + REDUNDANT_DROPS

# Drop only columns that exist in df
to_drop = [col for col in ALL_DROPS if col in df.columns]
df = df.drop(columns=to_drop)

print(f"Columns dropped: {len(to_drop)}")
print(f"Columns remaining: {df.shape[1]}")
print(f"\nRemaining columns:")
print(df.columns.tolist())

---
## 4. Feature Analysis

This section takes a closer look at the features, which will further inform which predictors to use in the project.

### 4.1 Categorical Features

In [ ]:
CATEGORICALS = [
    # Core acquisition signals
    'service_offering',

    # Ad platform attribution
    'campaign_group',
    'ad_network',

    # Geography
    'client_country',

    # Firmographic data
    'email_type',
    'cb_jobtitle_type',
    'cb_employees_range',
    'cb_annual_revenue_ranges',
    'cb_sector',

    # Lifecycle
    'lifecycle'
]

summarize_categoricals(df, CATEGORICALS)

for col in CATEGORICALS:
    plot_categorical(df, col)

**Impressions:** The reduced feature set consists of 10 categorical features and 1 numeric feature. Key observations from the categorical summary:

`ad_network` is dominated by `adwords` (63.9%) and `facebook` (32.0%), which together account for 95.9% of paid inbound customers. `bing` (2.1%), `linkedin` (0.9%), `-` (0.7%), and `twitter` (0.4%) are low-frequency values that may need to be grouped into an `other` bucket in preprocessing.

`service_offering` shows 8 unique values but is dominated by `virtual_assistant` (58.3%) and `executive_assistant` (27.3%), with the remaining 6 values collectively accounting for only 14.4% of records. These low-frequency categories will be grouped into an `other` bucket in preprocessing.

`campaign_group` has 5 unique values with `core_search` dominating at 58.9%, followed by `scaling` at 32.0%. Low-frequency categories will be reviewed for grouping in preprocessing.

`client_country` is heavily concentrated in `united_states` (85.0%) and `canada` (7.5%), with `philippines` at 4.7%. The remaining 11 countries each account for 0.2–0.6% of records and will be collapsed to `other` in preprocessing.

`email_type` is a clean binary split: `paid_email` (86.2%) vs. `free_email` (13.8%). No preprocessing needed beyond encoding.

`cb_jobtitle_type` is also binary: `senior_leadership` (64.9%) vs. `non_senior_leadership` (35.1%). Clean and balanced enough to encode as-is.

`cb_employees_range` is dominated by `1-10` (37.2%) with 30.5% missing. The distribution is heavily skewed toward small companies, with enterprise-sized companies (1k+) collectively accounting for under 2% of records. Low-frequency large-company bands may be grouped in preprocessing.

`cb_annual_revenue_ranges` has 31.2% missing. Among non-null records, the distribution is concentrated in `0-1m` (29.2%) and `1m-10m` (28.6%), with enterprise revenue bands ($500m+) collectively accounting for under 2% of records. Low-frequency bands will be grouped in preprocessing.

`cb_sector` has 19.6% missing. The top sectors are `consumer_discretionary` (24.9%), `industrials` (23.0%), `financials` (12.0%), and `information_technology` (10.1%). Low-frequency sectors (`telecommunication_services`, `materials`, `energy`, `utilities`) collectively account for under 2% and will be grouped into `other` in preprocessing.

`lifecycle` has 4 unique values and is heavily imbalanced: lifecycle 1 accounts for 89.2% of records, lifecycle 2 for 10.1%, and lifecycles 3 and 5 together for under 1%. Lifecycle 4 appears absent in the paid inbound population. This imbalance will be noted as a risk in the modeling phase — the model will have very limited exposure to repeat-engagement customers.

### 4.2 Numeric Features

In [ ]:
plot_numeric(df, 'starting_mrr')

**Impressions:** `starting_mrr` is the only numeric feature in the reduced set after dropping leaky features. As documented in Section 3.3.1, the majority of values reflect standard plan tiers with a clean base rate, but there are outliers extending to ~$93,571 that warrant further investigation. Most notably, row 194 shows an anomalously low `revenue_first_12_months` relative to its `starting_mrr`, and row 128 shows the inverse pattern. Both will be investigated in the preprocessing phase.

### 4.3 Feature Observations

**Feature Observations:** The reduced feature set consists of 10 categorical features and 1 numeric feature (`starting_mrr`). The near-absence of numeric features means the feature space will be dominated by OHE binary columns after encoding, which has implications for model selection that will be explored in the data geometry check below.

Several categorical features also carry meaningful missingness (`cb_annual_revenue_ranges` at 31.2%, `cb_employees_range` at 30.5%, `cb_sector` at 19.6%) and class imbalance (`lifecycle`, `ad_network`, `client_country`). These will require deliberate encoding and imputation strategies in the preprocessing phase.

### 4.4 Data Geometry
This section checks how the data geometry will look like post one hot encoding (OHE). This will give a clearer picture on which models may be suitable for the project.

In [ ]:
# Get reduced candidate features
CANDIDATE_FEATURES = [col for col in df.columns if col != 'revenue_first_12_months']

summary, sparsity = compute_geometry(CANDIDATE_FEATURES, df)
plot_design_matrix(CANDIDATE_FEATURES, df)

### 4.5 Data Geometry Observations

The reduced feature set of 11 features expands to **73 columns** after one-hot encoding, with an estimated sparsity of **84.9%**. This means that for any given customer record, approximately 15% of encoded feature values are non-zero.

The density in the design matrix heatmap is driven primarily by `client_country` (15 unique values, 23 OHE columns) and the firmographic features (`cb_annual_revenue_ranges`, `cb_sector`, `cb_employees_range`).

The geometry can be characterized as **sparse and categorical**. Tree-based models are the most natural fit for this structure — they make axis-aligned splits on binary indicator columns without being sensitive to scale or sparsity. Distance-based models like kNN are less suitable as the encoded space is dominated by category mismatches across binary columns.

---
## 5. Bivariate Analysis: Features vs. Target

This is the most important section for hypothesis generation. We want to understand whether each feature is associated with differences in CLV.

### 5.1 Categorical Features vs. Target `revenue_first_12_months`

For each categorical feature in the reduced candidate variables, we plot the distribution of 12-month revenue using box plots (spread) and median bar charts (central tendency). Groups are sorted by median CLV descending.

To rank features by signal strength prior to plotting, we compute a **median spread ratio** for each feature — the highest group median divided by the lowest group median. A ratio of 2.0x means the best-performing group has a median CLV twice that of the worst-performing group. The min and max group medians are shown alongside to give the absolute dollar gap. Features are ranked by spread ratio descending.

In [ ]:
TARGET = 'revenue_first_12_months'

CAT_FEATURES = [
    'ad_network',
    'service_offering',
    'campaign_group',
    'client_country',
    'email_type',
    'cb_jobtitle_type',
    'cb_employees_range',
    'cb_annual_revenue_ranges',
    'cb_sector',
    'lifecycle',
]

cat_summary = categorical_clv_summary(df, CAT_FEATURES, TARGET)
print(cat_summary.to_string(index=False))

**Observations:**

| Feature | Groups | Spread Ratio | Note |
|---|---|---|---|
| `client_country` | 14 | 4.25x | High cardinality; see plot for grouped analysis. |
| `cb_employees_range` | 8 | 3.18x | Strong signal; plot will show which company size bands drive the highest CLV. |
| `cb_annual_revenue_ranges` | 9 | 3.03x | Strong firmographic signal; plot will show which revenue bands drive the highest CLV. |
| `service_offering` | 8 | 2.71x | One of the stronger signals. This will tell us what service offering was promoted to the client. |
| `lifecycle` | 4 | 2.34x | Repeat customers expected to show higher median CLV. Sample counts per level will be checked for reliability given heavy imbalance toward lifecycle 1. |
| `campaign_group` | 5 | 2.06x | Moderate separation across campaign types. |
| `cb_sector` | 10 | 2.05x | Moderate signal; low-n sectors will be flagged for grouping in preprocessing. |
| `email_type` | 2 | 1.80x | Clean two-group comparison; plot will confirm direction and size of the gap. |
| `ad_network` | 6 | 1.77x | Moderate separation; adwords and facebook dominate volume (95.9% combined). Low-frequency platforms will be grouped into `other` in preprocessing. |
| `cb_jobtitle_type` | 2 | 1.46x | Weakest signal in the set. Will be retained through baseline but is a candidate for dropping. |

#### Feature: `ad_network`
Data dictionary definition: Ad network at acquisition

In [ ]:
plot_clv_by_category(df, 'ad_network', TARGET)

**Observation:** `adwords` dominates volume (n=342, 63.9%) but has the lowest median CLV among the main platforms at `$16,586`. `facebook` (n=171) shows a notably higher median at `$20,171`, suggesting it attracts higher-value customers despite lower volume. `bing` (n=11) shows a comparable median to `facebook` at $19,991 but with a much smaller sample. `linkedin` (n=5) and `twitter` (n=2) are too small to be reliable and will be grouped into `other` in preprocessing. The `-` value (n=4) represents records with an unresolved ad network attribution: these will be investigated in preprocessing to determine whether they can be mapped to a known platform before making a decision. The divergence between `adwords` and `facebook` median CLV is the key finding here and directly informs budget allocation decisions.

#### Feature: `service_offering`
Data dictionary definition: Service the client was interested in based on the ad/page they saw

In [ ]:
plot_clv_by_category(df, 'service_offering', TARGET)

**Observation:** `virtual_assistant` (n=312) and `executive_assistant` (n=146) dominate the paid inbound population, together accounting for 87.6% of records. `executive_assistant` (`$27,455` median) shows a meaningfully higher median CLV than `virtual_assistant` (`$14,386`) — a 1.91x gap — consistent with the expectation that a higher-tier service attracts higher-value customers. `brand` (n=23, `$31,920`) and `outsourcing` (n=9, `$31,020`) show higher medians than `executive_assistant` but with small sample sizes that make these estimates unreliable. `bookkeeping_accounting` (n=1) and `customer_support` (n=1) are single-record categories and will be dropped or grouped. `marketing` (n=7) is the lowest-performing category at $11,800. Low-frequency categories (`bookkeeping_accounting`, `customer_support`, `outsourcing`, `marketing`) will be reviewed for grouping into `other` in preprocessing.

#### Feature: `campaign_group`
Data dictionary definition: Campaign group classification at acquisition

In [ ]:
plot_clv_by_category(df, 'campaign_group', TARGET)

**Observation:** `core_search` dominates volume (n=315, 58.9%) but has the second lowest median CLV at `$16,449`. `branded` (n=23) leads with the highest median at `$31,920`. This is consistent with the expectation that branded search captures high-intent customers already familiar with the service. `scaling` (n=171) is the second largest group and shows a meaningful CLV advantage over `core_search` at `$20,171` vs. `$16,449`. `pmax` (n=15) sits in the middle at `$17,609`. `no_campaign_group` (n=11) has the lowest median at `$15,476` and likely represents unattributed or miscategorized records. This is worth investigating in preprocessing. The spread ratio of 2.06x is driven largely by the `branded` vs. `core_search` gap, which has a direct implication for budget allocation: branded campaigns attract disproportionately higher-value customers relative to their volume.

#### Feature: `client_country`
Data dictionary definition: Client's country derived from IP address

Given the cardinality of `client_country` (14 unique values in the paid inbound population), countries with n < 10 are grouped into `other` for visualization. The `client_country_grouped` grouping will carry forward as the encoding basis in preprocessing.

In [ ]:
# Group low-volume countries into 'other' for visualization
country_counts = df['client_country'].value_counts()
top_countries = country_counts[country_counts >= 10].index

df['client_country_grouped'] = df['client_country'].apply(
    lambda x: x if x in top_countries else 'other'
)

print(df['client_country_grouped'].value_counts())
print(f"\nCountries grouped into 'other': {(country_counts < 10).sum()}")

plot_clv_by_category(df, 'client_country_grouped', TARGET)

**Observation:** 11 low-volume countries (n < 10) are grouped into `other` for analysis, leaving 4 groups: `united_states` (n=455), `canada` (n=40), `philippines` (n=25), and `other` (n=15). `philippines` leads with the highest median at `$19,645`, followed by `united_states` at `$18,421`, `other` at `$16,449`, and `canada` at `$13,011`. The US-Canada gap is notable given that both are core markets; Canadian customers appear to generate meaningfully lower CLV than US customers. The `client_country_grouped` grouping applied here will carry forward as the basis for encoding in preprocessing.

#### Feature: `email_type`
Data dictionary definition: Classifies whether the contact is using a free email provider or a paid email (indicating company emails)

In [ ]:
plot_clv_by_category(df, 'email_type', TARGET)

**Observation:** `paid_email` (n=461, 86.2%) shows a substantially higher median CLV at `$19,819` vs. `$11,006` for `free_email` (n=74), which is a 1.80x gap. This is consistent with the expectation that corporate email users represent more serious B2B buyers with larger budgets. Despite being a simple binary feature, it provides clean and interpretable signal at minimal encoding cost.

#### Feature: `cb_jobtitle_type`
Data dictionary definition: Job title seniority classification. Values: senior leadership (C-suite, VP, Director), non-senior leadership

In [ ]:
plot_clv_by_category(df, 'cb_jobtitle_type', TARGET)

**Observation:** `senior_leadership` (n=347, 64.9%) shows a higher median CLV at `$21,080` vs. `$14,479` for `non_senior_leadership` (n=188), which is a 1.46x gap. The direction is consistent with the expectation that C-suite, VP, and Director-level buyers have larger budgets and higher service commitments. However, this is the weakest spread ratio in the categorical set, suggesting job title seniority alone is not a strong differentiator of CLV in the paid inbound population. It will be retained through the baseline model but remains a candidate for dropping if it contributes little to model performance.

#### Feature: `cb_employees_range`
Data dictionary definition: Company employee count range from Clearbit (e.g. 11-50, 51-250)

In [ ]:
plot_clv_by_category(df, 'cb_employees_range', TARGET)

**Observation:** `cb_employees_range` shows a non-monotonic relationship with CLV that is consistent with known business dynamics: mid-sized companies are the sweet spot for the service. `1k-5k` (n=4, `$26,047`), `51-250` (n=36, `$24,230`), and `251-1k` (n=15, `$23,549`) lead the CLV ranking, while the smallest band `1-10` (n=199) sits in the middle at `$17,007`. The largest bands `10k-50k` (n=2) and `5k-10k` (n=3) show the lowest medians but with sample sizes too small to be reliable. `100k+` (n=1) will be grouped in preprocessing. The non-monotonic pattern confirms this feature should be OHE-encoded rather than treated as ordinal. Low-frequency large-company bands will be reviewed for grouping in preprocessing.

#### Feature: `cb_annual_revenue_ranges`
Data dictionary definition: Company annual revenue range from Clearbit (e.g. `$1M-$10M`)

In [ ]:
plot_clv_by_category(df, 'cb_annual_revenue_ranges', TARGET)

**Observation:** `cb_annual_revenue_ranges` shows a similar non-monotonic pattern to `cb_employees_range`, again consistent with the known mid-market sweet spot. `$250m-$500m` (n=1, `$33,480`) leads but is a single record and unreliable. Among bands with meaningful sample sizes, `$50m-$100m` (n=7, `$23,549`) and `$1m-$10m` (n=153, `$20,163`) are the strongest performers, while the smallest company band `$0-$1m` (n=156, `$16,879`) sits lower despite being the second largest group. The very largest companies (`$10b+`, n=3; `$500m-$1b`, n=2) show lower medians, further reinforcing that enterprise-scale companies are not the core value segment. Low-frequency bands (`$250m-$500m`, `$500m-$1b`, `$1b-$10b`, `$10b+`, `$100m-$250m`) will be reviewed for grouping in preprocessing. Like `cb_employees_range`, this feature should be OHE-encoded rather than treated as ordinal given the non-monotonic pattern.

#### Feature: `cb_sector`
Data dictionary definition: Company sector from Clearbit — higher level than industry

In [ ]:
plot_clv_by_category(df, 'cb_sector', TARGET)

**Observation:** `cb_sector` shows moderate separation across 10 sectors. `energy` (n=2, `$34,865`) leads but with only 2 records is unreliable. Among sectors with meaningful sample sizes, `information_technology` (n=54, `$21,371`) and `health_care` (n=31, `$21,486`) are the stronger performers. `consumer_discretionary` (n=133) is the largest sector but has the lowest median at `$17,040`, while `industrials` (n=123) and `financials` (n=64) sit just above at `$17,979` and `$17,709` respectively. The three largest sectors are tightly clustered in median CLV, suggesting sector alone may not be a strong differentiator among the majority of the paid inbound population. Low-frequency sectors (`energy`, `telecommunication_services`, `materials`, `utilities`) will be grouped into `other` in preprocessing.

#### Feature: `lifecycle`
Data dictionary definition: The client's lifecycle count. Resets when a client churns or fails to convert within a 30-day window

It is cast to string to prevent ordinal misinterpretation by the model.

In [ ]:
plot_clv_by_category(df, 'lifecycle', TARGET)

**Observation:** `lifecycle` shows a counter-intuitive pattern — first-time customers (lifecycle 1, n=477, `$18,498`) outperform returning customers (lifecycle 2, n=54, `$14,168`) in median CLV, which challenges the expectation that repeat engagements generate higher value. Lifecycle 3 (n=3, `$13,802`) continues the downward trend, while lifecycle 5 (n=1, `$32,349`) is a single record and unreliable. Lifecycle 4 is absent from the paid inbound population entirely. The heavy imbalance toward lifecycle 1 (89.2%) means the model will have very limited exposure to repeat-engagement patterns, making any learned signal from lifecycle 2+ potentially unreliable. Lifecycles 3 and 5 will be grouped together in preprocessing given their very small sample sizes.

### 5.2 Numeric Feature vs. `revenue_first_12_months`

`starting_mrr` is the only numeric feature in the reduced set. The scatter plot shows the raw relationship, and Spearman ρ is used instead of Pearson r because the target is right-skewed (Section 2.1) and the relationship may be monotonic but non-linear.

In [ ]:
plot_clv_scatter(df, 'starting_mrr', TARGET)

valid_mrr = df[['starting_mrr', TARGET]].dropna()
rho, _ = spearmanr(valid_mrr['starting_mrr'], valid_mrr[TARGET])
print(f"Spearman ρ = {rho:.3f}")

**Observation:** `starting_mrr` shows a strong positive monotonic relationship with 12-month CLV, with a Spearman ρ of 0.856 — the strongest signal in the entire feature set by a wide margin. The scatter plot confirms the directional relationship: higher plan MRR at acquisition consistently associates with higher 12-month revenue. The Pearson r of 0.458 shown in the plot is lower due to the right-skewed distribution and the presence of outliers, which Spearman handles more robustly. The concentration of points in the lower-left corner reflects the dominance of standard plan tiers, while the sparse high-MRR outliers warrant treatment in preprocessing. This feature is expected to be the strongest single predictor in the regression model.

### 5.3 Signal Summary

Consolidates the median spread ratios for categorical features and the Spearman ρ for `starting_mrr` into a single reference table. Note that the two metrics are not directly comparable. Spread ratio measures group separation for categorical features, while Spearman ρ measures monotonic correlation for the numeric feature. Despite appearing at the bottom of the sorted table, `starting_mrr` (ρ = 0.856) is the strongest signal in the feature set by a wide margin. Among categorical features, `client_country`, `cb_employees_range`, and `cb_annual_revenue_ranges` show the greatest group separation.

In [ ]:
mrr_row = pd.DataFrame([{
    'Feature': 'starting_mrr',
    'Signal Metric': 'Spearman ρ',
    'Value': round(rho, 3)
}])

cat_rows = cat_summary[['Feature', 'Spread Ratio']].copy()
cat_rows = cat_rows.rename(columns={'Spread Ratio': 'Value'})
cat_rows['Signal Metric'] = 'Median Spread Ratio'

signal_table = pd.concat([mrr_row, cat_rows], ignore_index=True)
signal_table = signal_table.sort_values('Value', ascending=False).reset_index(drop=True)

print(signal_table.to_string(index=False))

### 5.4 Bivariate Observations & Modeling Hypotheses

**Overall observations**

The bivariate analysis confirms that most features in the reduced set are associated with meaningful differences in 12-month CLV. `starting_mrr` is the dominant signal by a wide margin (Spearman ρ = 0.856), consistent with the intuition that plan size at acquisition is the strongest predictor of 12-month value. Among categorical features, the firmographic variables — `cb_employees_range`, `cb_annual_revenue_ranges`, and `service_offering` — show the strongest group separation, while `cb_jobtitle_type` and `ad_network` are the weakest. A recurring theme across firmographic features is a non-monotonic relationship with CLV: mid-sized companies by both employee count and annual revenue outperform both the smallest and largest segments, which is consistent with known business dynamics. Features with low-frequency categories (`service_offering`, `cb_sector`, `cb_employees_range`, `cb_annual_revenue_ranges`, `ad_network`, `client_country`) will require grouping strategies in preprocessing before encoding.

---

**Modeling hypotheses**

- **H1 (Starting MRR):** `starting_mrr` will be the strongest single predictor of 12-month CLV. As a direct proxy for plan size at acquisition, it captures committed spend before any behavioral signal is available.

- **H2 (Service offering):** `executive_assistant` customers will generate materially higher CLV than `virtual_assistant` customers, reflecting a higher-tier product and price point. The 1.91x median gap observed in the bivariate analysis supports this.

- **H3 (Firmographic mid-market signal):** Firmographic features (`cb_employees_range`, `cb_annual_revenue_ranges`) will provide incremental signal beyond plan size alone, specifically identifying mid-market companies as the highest-value segment. The non-monotonic pattern observed in both features makes OHE the appropriate encoding strategy.

- **H4 (Campaign type):** `branded` campaign customers will show higher CLV than `core_search` customers, reflecting that branded search captures higher-intent buyers already familiar with the service.

- **H5 (Lifecycle):** The model will find limited signal from `lifecycle` given the severe imbalance toward lifecycle 1 (89.2%). Repeat-engagement patterns will be difficult to learn reliably from the available data.

---
## 6. Segment Analysis

Section 5 examined each feature independently against the target. This section examines whether combinations of features reveal patterns that univariate analysis misses — specifically, whether the CLV advantage of one feature holds consistently across the groups of another.

Features are selected based on two criteria: spread ratio from Section 5.3 and direct relevance to paid advertising decisions. `client_country` is excluded as 85% of records collapse into a single group after grouping, leaving insufficient variation for reliable interaction cells.

| Feature | Spread Ratio | Dollar Gap |
|---|---|---|
| `cb_employees_range` | 3.18x | `$17,847` |
| `cb_annual_revenue_ranges` | 3.03x | `$22,434` |
| `service_offering` | 2.71x | `$20,120` |
| `lifecycle` | 2.34x | `$18,547` |
| `campaign_group` | 2.06x | `$16,444` |
| `ad_network` | 1.77x | `$10,335` |

The six features above yield 15 pairwise combinations to screen.

In [ ]:
INTERACTION_FEATURES = [
    'cb_employees_range',
    'cb_annual_revenue_ranges',
    'service_offering',
    'lifecycle',
    'campaign_group',
    'ad_network',
]

interactions = pairwise_interaction(df, INTERACTION_FEATURES, TARGET)
print(interactions.to_string(index=False))

---
## 7. EDA Summary & Modeling Implications

This section is the EDA memo. It synthesizes findings into decisions that carry forward into preprocessing and modeling.